In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected!")

In [ ]:
df = session.table("INSURANCE_ML.ML_PIPE.INSURANCE_DATA").to_pandas()

print("Data loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
X = df.drop("CHARGES", axis=1)
y = df["CHARGES"]

print("Features (X):", X.shape)
print("Target (y):", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_features = ["AGE", "BMI", "CHILDREN"]
categorical_features = ["SEX", "SMOKER", "REGION"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessor created successfully!")

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            random_state=42
        ))
    ]
)

print("XGBoost model pipeline created successfully!")

In [ ]:
model_pipeline.fit(X_train, y_train)

print("Model training completed successfully!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Results")
print("------------------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="INSURANCE_ML",
    schema_name="ML_PIPE"
)

print("Snowflake Model Registry connected successfully!")

In [ ]:
from snowflake.ml.model import target_platform

model_ref = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    version_name="V1",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    target_platforms=target_platform.WAREHOUSE_ONLY,
    pip_requirements=[
        "scikit-learn",
        "xgboost"
    ],
    artifact_repository_map={
        "pip": "snowflake.snowpark.pypi_shared_repository"
    },
    comment="XGBoost regression model for predicting insurance charges - Warehouse inference enabled."
)

print("Model registered successfully!")
print("Model name: INSURANCE_CHARGES_MODEL")
print("Version: V1")

In [ ]:
session.sql("USE WAREHOUSE ML_PIPELINE_WH").collect()
session.sql("USE DATABASE INSURANCE_ML").collect()
session.sql("USE SCHEMA ML_PIPE").collect()

print("Database context restored successfully!")

In [ ]:
from snowflake.ml.model import target_platform

model_ref = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    version_name="V1",
    sample_input_data=X_train,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    target_platforms=target_platform.WAREHOUSE_ONLY,
    pip_requirements=[
        "scikit-learn",
        "xgboost"
    ],
    artifact_repository_map={
        "pip": "snowflake.snowpark.pypi_shared_repository"
    },
    comment="XGBoost regression model for predicting insurance charges - Warehouse inference enabled."
)

print("Model registered successfully!")
print("Model name: INSURANCE_CHARGES_MODEL")
print("Version: V1")

In [ ]:
model_v1 = registry.get_model("INSURANCE_CHARGES_MODEL").version("V1")

print("Registered model loaded successfully!")

In [ ]:
registered_predictions = model_v1.run(
    X_test,
    function_name="predict"
)

print("Warehouse inference completed successfully!")
registered_predictions.head()

In [ ]:
USE WAREHOUSE ML_PIPELINE_WH;
USE DATABASE INSURANCE_ML;
USE SCHEMA ML_PIPE;

CREATE OR REPLACE TABLE INCOMING_DATA (
    AGE INTEGER,
    SEX VARCHAR,
    BMI FLOAT,
    CHILDREN INTEGER,
    SMOKER VARCHAR,
    REGION VARCHAR
);

SELECT *
FROM INCOMING_DATA;

In [ ]:
USE WAREHOUSE ML_PIPELINE_WH;
USE DATABASE INSURANCE_ML;
USE SCHEMA ML_PIPE;

CREATE OR REPLACE TABLE INCOMING_DATA (
    AGE INTEGER,
    SEX VARCHAR,
    BMI FLOAT,
    CHILDREN INTEGER,
    SMOKER VARCHAR,
    REGION VARCHAR
);

SELECT *
FROM INCOMING_DATA;